## Academic Approach to Machine Learning for Medical Imaging
**Level: Beginner**

Building a machine learning model that works is different from building one that is
scientifically valid, reproducible, and defensible as research. This notebook covers
the habits and checks that separate a "working demo" from academically rigorous work —
using examples from our own prior classification and segmentation notebooks where relevant.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit

### 1. Formulating a Research Question

A vague goal like *"I want to classify chest X-rays"* isn't a research question — it's a
task description. An academic framing asks: *what specific, testable claim am I making,
and how would I know if it's false?*

**Example — weak vs. strong framing:**
- Weak: "Build a model to classify medical images."
- Strong: "Can a DenseNet121 model, trained on MedNIST, distinguish six imaging modalities
  with performance comparable to a simple baseline, and does data augmentation
  meaningfully improve generalization on the held-out test set?"

The strong version is falsifiable — it specifies a model, a comparison point, and a
measurable outcome.

### 2. Dataset Considerations and Data Leakage

One of the most common mistakes in medical imaging ML is **data leakage between train
and test sets** — specifically, splitting by *image* instead of by *patient*. If a
patient has multiple scans, and some end up in training while others end up in test,
the model can effectively "memorize" that patient rather than learning to generalize,
silently inflating reported performance.

The code below simulates this with dummy patient data.

In [ ]:
# Simulate a dataset: 20 patients, each with 5 images (100 images total)
patient_ids = np.repeat(np.arange(20), 5)
image_ids = np.arange(100)

df = pd.DataFrame({"image_id": image_ids, "patient_id": patient_ids})

# WRONG: naive random split by image — ignores that some patients
# will end up with images in BOTH train and test
train_wrong, test_wrong = train_test_split(df, test_size=0.2, random_state=0)

leaked_patients = set(train_wrong["patient_id"]) & set(test_wrong["patient_id"])
print(f"Patients appearing in BOTH train and test (image-level split): {len(leaked_patients)}")

In [ ]:
# RIGHT: split by patient first, so all of a patient's images stay together
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=0)
train_idx, test_idx = next(splitter.split(df, groups=df["patient_id"]))

train_correct = df.iloc[train_idx]
test_correct = df.iloc[test_idx]

leaked_patients_fixed = set(train_correct["patient_id"]) & set(test_correct["patient_id"])
print(f"Patients appearing in BOTH train and test (patient-level split): {len(leaked_patients_fixed)}")

### 3. Baselines and Fair Comparison

A reported metric means little in isolation. Academic work always compares a new model
against a **baseline** — a simple model, random-chance performance, or a prior published
result.

**Example:** our classification notebook achieved 99.98% test accuracy. Is that
impressive? Only relative to a baseline:
- Random guessing across 6 classes would score ~16.7%
- A simple baseline (e.g., logistic regression on raw pixels) might score meaningfully
  lower than a CNN
- MedNIST's classes are visually distinct by design, so even a simple baseline might
  score quite high — which is itself an important, honest caveat to report

Without stating a baseline, a single accuracy number can't be judged as "good" or "bad."

### 4. Statistical Validity

The same accuracy score means different things depending on sample size. 99% accuracy
on 50 images carries far more uncertainty than 99% on 5,895 images. Academic reporting
typically includes a confidence interval, not just a point estimate.

In [ ]:
from scipy.stats import norm

def accuracy_confidence_interval(accuracy, n, confidence=0.95):
    z = norm.ppf(1 - (1 - confidence) / 2)
    se = np.sqrt((accuracy * (1 - accuracy)) / n)
    return accuracy - z * se, accuracy + z * se

# Compare the SAME accuracy at two different sample sizes
for n in [50, 5895]:
    low, high = accuracy_confidence_interval(0.9998, n)
    print(f"n={n}: 95% CI = [{low:.4f}, {high:.4f}]")

### 5. Reproducibility

Academic work must be repeatable by others. Key practices:
- **Set random seeds** — both our prior notebooks used `set_determinism(seed=0)` for
  exactly this reason
- **Document library versions** — so results can be reproduced on the same software stack
- **Share code and data availability** — or clearly state restrictions if data can't be shared

In [ ]:
!pip install -q monai

import torch, monai
print("PyTorch version:", torch.__version__)
print("MONAI version:", monai.__version__)

### 6. Reading a Real Paper — Worked Example

**Paper:** *"Inflation of test accuracy due to data leakage in deep learning-based
classification of OCT images"* (PMC, 2022)
https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9500039/

This paper studies the exact issue we simulated in Section 2 — but with real data.
The authors compared model performance under two splitting strategies across three
real, widely-used OCT (eye scan) datasets:

- **Improper split** — images divided randomly, allowing images from the same patient
  to land in both train and test sets
- **Proper split** — images divided by patient first, so no patient appears in both sets

**Key finding:** improper splitting inflated reported test accuracy by roughly
**5–30%**, depending on the dataset — meaning some published results in this space may
be substantially overstating real-world performance.

**Discussion questions for students:**
- Why might improper splitting inflate performance *more* for some datasets than others?
- If you saw a paper reporting 99% accuracy with no mention of how data was split,
  what would you want to check before trusting that number?
- How does this connect to the "Baselines and Fair Comparison" section above — why
  isn't the number alone ever enough?

### 7. Common Pitfalls in Medical Imaging ML

- **Patient-level data leakage** (covered above)
- **Overfitting on small datasets** — deep models can memorize small datasets easily
- **Lack of external validation** — a model validated only on one hospital's data may
  not generalize to a different scanner or population
- **Dataset bias** — demographic or equipment imbalances in training data can silently
  bias model performance

### Try It Yourself

1. Here is a flawed study design: *"We split 500 chest X-rays randomly into train/test,
   achieving 98% test accuracy."* What's missing to know if this is trustworthy?
2. Take the strong research question from Section 1 — rewrite it for a segmentation task
   instead of classification.
3. Using the code from Section 2, simulate a dataset with 100 patients and 3 images each.
   How many patients leak with a naive split?
4. Why might a model with 99% accuracy on one hospital's data perform much worse at a
   different hospital? List two possible causes.